# PREPARATION DES DONNEES

## Récupération des données via l'API NASA POWER

In [ ]:
import requests
import pandas as pd

url = "https://power.larc.nasa.gov/api/temporal/daily/point"
params = {
    "parameters": "PRECTOTCORR,T2M",
    "community": "AG",
    "longitude": -1.52,
    "latitude": 12.37,
    "start": "20000101",
    "end": "20241231",
    "format": "JSON"
}

response = requests.get(url, params=params)
data = response.json()

# Les données sont nichées dans data["properties"]["parameter"]
print(data["properties"]["parameter"].keys())

## Nettoyage de base

In [ ]:
df = pd.DataFrame.from_dict(data["properties"]["parameter"], orient="index")
df = df.transpose()  # Transpose pour avoir les dates en index et les paramètres en colonnes
df = df.reset_index().rename(columns={"index": "date", "PRECTOTCORR": "precipitation", "T2M": "temperature"})
print(df.head())


df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
df["precipitation"] = pd.to_numeric(df["precipitation"])
df["temperature"] = pd.to_numeric(df["temperature"])

print(df.info())
print((df["precipitation"] == -999).sum(), "valeurs manquantes en précipitation")
print((df["temperature"] == -999).sum(), "valeurs manquantes en température")

# ANALYSE EXPLORATOIRE (EDA)

1. Saisonalité annuelle

In [ ]:
import matplotlib.pyplot as plt

df["annee"] = df["date"].dt.year
df["mois"] = df["date"].dt.month

precip_annuelle = df.groupby("annee")["precipitation"].sum()

plt.figure(figsize=(12,5))
plt.plot(precip_annuelle.index, precip_annuelle.values, marker="o")
plt.title("Précipitations totales annuelles — Ouagadougou")
plt.xlabel("Année")
plt.ylabel("Précipitations (mm)")
plt.grid(True)
plt.show()

2. Saisonalité mensuelle

In [ ]:
# Étape 1 : cumul mensuel PAR année
cumul_mensuel_par_annee = df.groupby(["annee", "mois"])["precipitation"].sum()

# Étape 2 : moyenne de ces cumuls, mois par mois, à travers les années
precip_mensuelle = cumul_mensuel_par_annee.groupby("mois").mean()

plt.figure(figsize=(12,5))
plt.bar(precip_mensuelle.index, precip_mensuelle.values, color="steelblue")
plt.title("Cumul pluviométrique mensuel moyen — Ouagadougou (2000-2024)")
plt.xlabel("Mois")
plt.ylabel("Précipitations (mm, cumul mensuel moyen)")
plt.xticks(range(1,13))
plt.grid(True, axis="y")
plt.show()

3. Relation précipitation / température

In [ ]:
# Moyenne mensuelle de température
temp_mensuelle = df.groupby("mois")["temperature"].mean()

fig, ax1 = plt.subplots(figsize=(12,5))

ax1.bar(precip_mensuelle.index, precip_mensuelle.values, color="steelblue", alpha=0.6, label="Précipitations")
ax1.set_xlabel("Mois")
ax1.set_ylabel("Précipitations (mm)", color="steelblue")

ax2 = ax1.twinx()
ax2.plot(temp_mensuelle.index, temp_mensuelle.values, color="red", marker="o", label="Température")
ax2.set_ylabel("Température (°C)", color="red")

plt.title("Précipitations et température mensuelles — Ouagadougou")
plt.xticks(range(1,13))
plt.show()

4. Bilan de l'EDA

- Forte variabilité interannuelle du cumul pluviométrique (500-1100 mm)
- Saisonnalité mensuelle très marquée (régime de mousson)
- Relation pluie/température complexe, pas linéaire sur l'année entière

# MODELISATION

1. Construction du jeu de données

In [ ]:
# Agrégats annuels
annuel = df.groupby("annee").agg(
    cumul_annee=("precipitation", "sum"),
    temp_moyenne_annee=("temperature", "mean")
).reset_index()

# La cible = le cumul de l'année suivante
annuel["cumul_annee_suivante"] = annuel["cumul_annee"].shift(-1)

print(annuel)

In [ ]:
# Supprimer la dernière ligne (2024) qui n'a pas de cible valide
annuel = annuel.dropna(subset=["cumul_annee_suivante"])
print(annuel)
print(f"\nNombre d'observations utilisables : {len(annuel)}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

X = annuel[["cumul_annee", "temp_moyenne_annee"]]
y = annuel["cumul_annee_suivante"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


2. Modélisation avec LinearRegression

In [ ]:
from sklearn.linear_model import LinearRegression

#Entraînement du modèle
model = LinearRegression()
model.fit(X_train, y_train)

#prediction et évaluation du modèle
y_pred = model.predict(X_test)
print("MAE :", mean_absolute_error(y_test, y_pred))
print("R² :", r2_score(y_test, y_pred))
print("\nPrédictions vs réel :")
for pred, reel in zip(y_pred, y_test):
    print(f"Prédit : {pred:.0f} mm | Réel : {reel:.0f} mm")

3. Modélisation avec RandomForest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

#entrainement du modèle Random Forest
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)

#prediction et évaluation du modèle Random Forest
y_pred_rf = model_rf.predict(X_test)

print("MAE (Random Forest) :", mean_absolute_error(y_test, y_pred_rf))
print("R² (Random Forest) :", r2_score(y_test, y_pred_rf))

print("\nPrédictions vs réel :")
for pred, reel in zip(y_pred_rf, y_test):
    print(f"Prédit : {pred:.0f} mm | Réel : {reel:.0f} mm")